# [2] 선거인수 정보 수집

**End Point**: `https://apis.data.go.kr/9760000/ElcntInfoInqireService`

## 레벨별 오퍼레이션 및 필수 파라미터

| 레벨 | 오퍼레이션 | 필수 파라미터 |
|---|---|---|
| 시도별 | `getCtpvElcntInfoInqire` | sgId |
| 구시군별 | `getGsigElcntInfoInqire` | sgId, sdName |
| 읍면동별 | `getEmdElcntInfoInqire` | sgId, sdName, wiwName |
| 투표구별 | `getVtdsElcntInfoInqire` | sgId, sdName, wiwName |
| 선거구별 | `getElpcElcntInfoInqire` | sgId, sgTypecode |

## 주요 응답 컬럼 (PDF 확인)

| 컬럼명 | 설명 |
|---|---|
| cfmtnElcnt | 확정선거인수(계) |
| cfmtnManElcnt | 확정선거인수(남) |
| cfmtnFmlElcnt | 확정선거인수(여) |
| ppltCnt | 인구수 |
| emdCount | 읍면동수 |
| tpgCount | 투표구수 |
| tpgName | 투표구명 (투표구별만) |
| emdName | 읍면동명 (읍면동별/투표구별) |
| sggName | 선거구명 (선거구별만) |
| cfmtnRdvtDccnt | 거소투표 신고인명부 등재자수(계) |

In [2]:
# ✏️ 본인 Decoding 키로 교체하세요
API_KEY = "6lVhhlLRaGq/+tidZgS0POWCcl7BOqRJXiDj+Xtl/+rJJVEqNPWFjwFpyWkOn3NaNqacOHvj9UG+BnHAGBFd4w=="

In [3]:
import requests
import pandas as pd
import time

def fetch_all_pages(base_url, fixed_params, page_size=100, delay=0.3):
    all_items = []
    page = 1
    while True:
        params = {**fixed_params, "pageNo": str(page), "numOfRows": str(page_size), "resultType": "json"}
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            if resp.status_code != 200:
                print(f"  ⚠️  HTTP {resp.status_code}: {resp.text[:300]}")
                break
            data = resp.json()
            header = data["response"]["header"]
            if header["resultCode"] not in ("INFO-00", "00"):
                print(f"  ⚠️  API 오류: {header['resultMsg']}")
                break
            body  = data["response"]["body"]
            total = int(body.get("totalCount", 0))
            raw   = body.get("items") or {}
            items = raw.get("item", []) if isinstance(raw, dict) else []
            if isinstance(items, dict):
                items = [items]
            all_items.extend(items)
            print(f"  페이지 {page}: {len(items)}건  (누적 {len(all_items)}/{total})")
            if len(all_items) >= total or not items:
                break
            page += 1
            time.sleep(delay)
        except Exception as e:
            print(f"  ❌ 오류 (페이지 {page}): {e}")
            break
    return all_items

def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n💾 저장 완료: {path}  ({len(df)}행 × {len(df.columns)}열)")

In [4]:
ELECTIONS = {
    "20200415": "제21대 국회의원선거",
    "20220309": "제20대 대통령선거",
    "20220601": "제8회 전국동시지방선거",
    "20231011": "제22대 국선 강서구청장 보궐선거",
    "20240410": "제22대 국회의원선거",
    "20250603": "제21대 대통령선거",
}

ELECTIONS_LOCAL = {
    "20140604": "제6회 전국동시지방선거",
    "20180613": "제7회 전국동시지방선거",
    "20220601": "제8회 전국동시지방선거",
}

In [6]:
BASE = "https://apis.data.go.kr/9760000/ElcntInfoInqireService"

# ── 수집 레벨 선택 ──────────────────────────────
# "시도별" / "구시군별" / "읍면동별" / "투표구별" / "선거구별"
QUERY_LEVEL = "읍면동별"
# ────────────────────────────────────────────────

# sgTypecode: 선거구별 조회 시 필요 (선거별로 다름)
# 1=대통령 2=국회의원 3=시도지사 4=구시군의장 5=시도의회 6=구시군의회 10=교육의원 11=교육감
SG_TYPE_MAP = {
    "20200415": "2",  # 국회의원
    "20220309": "1",  # 대통령
    "20220601": "3",  # 지방(시도지사 대표)
    "20231011": "4",  # 보궐(구시군의장)
    "20240410": "2",  # 국회의원
    "20250603": "1",  # 대통령
}

SD_NAMES = [
    "서울특별시", "부산광역시", "대구광역시", "인천광역시", "광주광역시",
    "대전광역시", "울산광역시", "세종특별자치시", "경기도", "강원특별자치도",
    "충청북도", "충청남도", "전북특별자치도", "전라남도", "경상북도",
    "경상남도", "제주특별자치도"
]

LEVEL_CONFIG = {
    "시도별":   {"url": f"{BASE}/getCtpvElcntInfoInqire",  "extra_keys": []},
    "구시군별": {"url": f"{BASE}/getGsigElcntInfoInqire",  "extra_keys": ["sdName"]},
    "읍면동별": {"url": f"{BASE}/getEmdElcntInfoInqire",   "extra_keys": ["sdName", "wiwName"]},
    "투표구별": {"url": f"{BASE}/getVtdsElcntInfoInqire",  "extra_keys": ["sdName", "wiwName"]},
    "선거구별": {"url": f"{BASE}/getElpcElcntInfoInqire",  "extra_keys": ["sgTypecode"]},
}

config = LEVEL_CONFIG[QUERY_LEVEL]
print(f"조회 레벨: {QUERY_LEVEL}")
print(f"URL: {config['url']}")
print(f"추가 필수 파라미터: {config['extra_keys']}")

조회 레벨: 읍면동별
URL: https://apis.data.go.kr/9760000/ElcntInfoInqireService/getEmdElcntInfoInqire
추가 필수 파라미터: ['sdName', 'wiwName']


In [8]:
GSIG_URL = f"{BASE}/getGsigElcntInfoInqire"
EMD_URL  = f"{BASE}/getEmdElcntInfoInqire"

all_records = []

for sg_id, election_name in ELECTIONS_LOCAL.items():
    print(f"\n===== {election_name} (sgId={sg_id}) =====")
    for sd in SD_NAMES:
        # 1) 해당 시도의 구시군 목록 조회
        gsig_items = fetch_all_pages(
            base_url=GSIG_URL,
            fixed_params={"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd},
        )
        wiw_names = sorted({item["wiwName"] for item in gsig_items if item.get("wiwName")})

        # 2) 구시군별로 읍면동 단위 선거인수 조회
        for wiw in wiw_names:
            items = fetch_all_pages(
                base_url=EMD_URL,
                fixed_params={"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd, "wiwName": wiw},
            )
            for item in items:
                item["election_name"] = election_name
            all_records.extend(items)
            if items:
                print(f"  {sd} {wiw}: {len(items)}건")


===== 제6회 전국동시지방선거 (sgId=20140604) =====
  페이지 1: 26건  (누적 26/26)
  페이지 1: 23건  (누적 23/23)
  서울특별시 강남구: 23건
  페이지 1: 19건  (누적 19/19)
  서울특별시 강동구: 19건
  페이지 1: 14건  (누적 14/14)
  서울특별시 강북구: 14건
  페이지 1: 21건  (누적 21/21)
  서울특별시 강서구: 21건
  페이지 1: 22건  (누적 22/22)
  서울특별시 관악구: 22건
  페이지 1: 16건  (누적 16/16)
  서울특별시 광진구: 16건
  페이지 1: 16건  (누적 16/16)
  서울특별시 구로구: 16건
  페이지 1: 11건  (누적 11/11)
  서울특별시 금천구: 11건
  페이지 1: 20건  (누적 20/20)
  서울특별시 노원구: 20건
  페이지 1: 15건  (누적 15/15)
  서울특별시 도봉구: 15건
  페이지 1: 15건  (누적 15/15)
  서울특별시 동대문구: 15건
  페이지 1: 16건  (누적 16/16)
  서울특별시 동작구: 16건
  페이지 1: 17건  (누적 17/17)
  서울특별시 마포구: 17건
  페이지 1: 15건  (누적 15/15)
  서울특별시 서대문구: 15건
  페이지 1: 19건  (누적 19/19)
  서울특별시 서초구: 19건
  페이지 1: 18건  (누적 18/18)
  서울특별시 성동구: 18건
  페이지 1: 21건  (누적 21/21)
  서울특별시 성북구: 21건
  페이지 1: 27건  (누적 27/27)
  서울특별시 송파구: 27건
  페이지 1: 19건  (누적 19/19)
  서울특별시 양천구: 19건
  페이지 1: 19건  (누적 19/19)
  서울특별시 영등포구: 19건
  페이지 1: 17건  (누적 17/17)
  서울특별시 용산구: 17건
  페이지 1: 17건  (누적 17/17)
  서울특별시 은평구: 17건
  페이지 

In [6]:
# all_records = []

# for sg_id, election_name in ELECTIONS.items():
#     print(f"\n{'='*55}")
#     print(f"📌 {election_name}  (sgId={sg_id})")
#     print(f"{'='*55}")

#     extra_keys = config["extra_keys"]

#     # 추가 파라미터 없이 바로 조회 가능한 레벨 (시도별, 선거구별)
#     if not extra_keys or extra_keys == ["sgTypecode"]:
#         fixed = {"serviceKey": API_KEY, "sgId": sg_id}
#         if "sgTypecode" in extra_keys:
#             fixed["sgTypecode"] = SG_TYPE_MAP.get(sg_id, "1")
#         items = fetch_all_pages(base_url=config["url"], fixed_params=fixed)
#         for item in items:
#             item["election_name"] = election_name
#         all_records.extend(items)
#         print(f"  ✅ {len(items)}건")

#     # sdName 순회가 필요한 레벨 (구시군별, 읍면동별, 투표구별)
#     elif "sdName" in extra_keys and "wiwName" not in extra_keys:
#         for sd in SD_NAMES:
#             fixed = {"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd}
#             items = fetch_all_pages(base_url=config["url"], fixed_params=fixed)
#             for item in items:
#                 item["election_name"] = election_name
#             all_records.extend(items)
#             if items:
#                 print(f"  {sd}: {len(items)}건")

#     # sdName + wiwName 모두 필요한 레벨 (읍면동별, 투표구별) — 데이터 많아 주의
#     else:
#         print("  ℹ️  읍면동별/투표구별은 wiwName이 필수입니다.")
#         print("  아래 예시처럼 특정 구시군만 조회하거나, 구시군 목록을 별도로 구성하세요.")
#         # 예시: 서울 종로구만 조회
#         fixed = {"serviceKey": API_KEY, "sgId": sg_id, "sdName": "서울특별시", "wiwName": "종로구"}
#         items = fetch_all_pages(base_url=config["url"], fixed_params=fixed)
#         for item in items:
#             item["election_name"] = election_name
#         all_records.extend(items)
#         print(f"  (예시) 서울 종로구: {len(items)}건")

# print(f"\n총 {len(all_records)}건 수집 완료")

In [11]:
if not all_records:
    print("⚠️  수집된 데이터 없음")
else:
    df = pd.DataFrame(extra_records)

    # PDF 확인된 실제 컬럼명 (전체 레벨 공통 + 레벨별 추가)
    col_map = {
        "election_name":        "선거명",
        "sgId":                 "선거ID",
        "sdName":               "시도명",
        "wiwName":              "구시군명",
        "sggName":              "선거구명",
        "emdName":              "읍면동명",
        "tpgName":              "투표구명",
        "wiwCount":             "구시군수",
        "emdCount":             "읍면동수",
        "tpgCount":             "투표구수",
        "ppltCnt":              "인구수",
        "ntabPpltCnt":          "인구수(재외국민)",
        "frgnrPpltCnt":         "인구수(외국인)",
        "cfmtnElcnt":           "확정선거인수(계)",
        "cfmtnRacnt":           "확정선거인수(재외국민)",
        "cfmtnFrgnrCnt":        "확정선거인수(외국인)",
        "cfmtnManElcnt":        "확정선거인수(남)",
        "cfmtnManRacnt":        "확정선거인수(남_재외국민)",
        "cfmtnManFrgnrCnt":     "확정선거인수(남_외국인)",
        "cfmtnFmlElcnt":        "확정선거인수(여)",
        "cfmtnFmlRacnt":        "확정선거인수(여_재외국민)",
        "cfmtnFmlFrgnrCnt":     "확정선거인수(여_외국인)",
        "cfmtnRdvtDccnt":       "거소투표신고인명부등재자수(계)",
        "cfmtnNtabRdvtDccnt":   "거소투표신고인명부등재자수(재외국민)",
        "cfmtnRdvtManDccnt":    "거소투표신고인명부등재자수(남)",
        "cfmtnNtabRdvtManDccnt":"거소투표신고인명부등재자수(남_재외국민)",
        "cfmtnRdvtFmlDccnt":    "거소투표신고인명부등재자수(여)",
        "cfmtnNtabRdvtFmlDccnt":"거소투표신고인명부등재자수(여_재외국민)",
        "num":                  "결과순서",
    }
    existing = [c for c in col_map if c in df.columns]
    df = df[existing].rename(columns=col_map)

    # 숫자형 변환
    num_cols = [c for c in df.columns if any(k in c for k in ["수", "인구"])]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    display(df.head(10))
    save_csv(df, f"002_선거인수정보_{QUERY_LEVEL}.csv")

,선거명,선거ID,시도명,구시군명,읍면동명,투표구수,인구수,인구수(재외국민),인구수(외국인),확정선거인수(계),...,확정선거인수(여),확정선거인수(여_재외국민),확정선거인수(여_외국인),거소투표신고인명부등재자수(계),거소투표신고인명부등재자수(재외국민),거소투표신고인명부등재자수(남),거소투표신고인명부등재자수(남_재외국민),거소투표신고인명부등재자수(여),거소투표신고인명부등재자수(여_재외국민),결과순서
0,제6회 전국동시지방선거,20140604,강원도,강릉시,합계,62,216508,172,126,176430,...,89885,95,80,648,0,387,0,261,0,1
1,제6회 전국동시지방선거,20140604,강원도,강릉시,주문진읍,6,18568,2,14,16098,...,8221,1,10,30,0,24,0,6,0,2
2,제6회 전국동시지방선거,20140604,강원도,강릉시,성산면,2,3369,8,3,2976,...,1467,4,2,10,0,8,0,2,0,3
3,제6회 전국동시지방선거,20140604,강원도,강릉시,왕산면,3,1738,0,1,1635,...,737,0,1,4,0,3,0,1,0,4
4,제6회 전국동시지방선거,20140604,강원도,강릉시,구정면,2,4015,4,0,3536,...,1769,3,0,18,0,6,0,12,0,5
5,제6회 전국동시지방선거,20140604,강원도,강릉시,강동면,2,5153,7,1,4615,...,2261,2,0,107,0,41,0,66,0,6
6,제6회 전국동시지방선거,20140604,강원도,강릉시,옥계면,2,4288,2,1,3815,...,1865,2,1,68,0,26,0,42,0,7
7,제6회 전국동시지방선거,20140604,강원도,강릉시,사천면,2,4454,7,3,3882,...,1859,3,3,7,0,3,0,4,0,8
8,제6회 전국동시지방선거,20140604,강원도,강릉시,연곡면,2,6964,4,2,5853,...,2953,4,2,11,0,8,0,3,0,9
9,제6회 전국동시지방선거,20140604,강원도,강릉시,홍제동,2,8528,5,14,6845,...,3508,3,9,11,0,8,0,3,0,10



💾 저장 완료: 002_선거인수정보_읍면동별.csv  (1388행 × 25열)


In [10]:
import os
os.getcwd()

'/Users/jamiejeong-eunpark/Downloads/Python_Project/Data&Code'

In [9]:
# 6~8회(2014~2022) 지방선거는 개편 전 도명으로 조회해야 함
MISSING_SD = ["강원도", "전라북도"]

extra_records = []
for sg_id, election_name in ELECTIONS_LOCAL.items():
    for sd in MISSING_SD:
        gsig_items = fetch_all_pages(
            base_url=GSIG_URL,
            fixed_params={"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd},
        )
        wiw_names = sorted({item["wiwName"] for item in gsig_items if item.get("wiwName")})
        for wiw in wiw_names:
            items = fetch_all_pages(
                base_url=EMD_URL,
                fixed_params={"serviceKey": API_KEY, "sgId": sg_id, "sdName": sd, "wiwName": wiw},
            )
            for item in items:
                item["election_name"] = election_name
            extra_records.extend(items)
            if items:
                print(f"  {sd} {wiw}: {len(items)}건")

print(f"\n추가 수집: {len(extra_records)}건")

  페이지 1: 19건  (누적 19/19)
  페이지 1: 22건  (누적 22/22)
  강원도 강릉시: 22건
  페이지 1: 6건  (누적 6/6)
  강원도 고성군: 6건
  페이지 1: 11건  (누적 11/11)
  강원도 동해시: 11건
  페이지 1: 13건  (누적 13/13)
  강원도 삼척시: 13건
  페이지 1: 9건  (누적 9/9)
  강원도 속초시: 9건
  페이지 1: 6건  (누적 6/6)
  강원도 양구군: 6건
  페이지 1: 7건  (누적 7/7)
  강원도 양양군: 7건
  페이지 1: 10건  (누적 10/10)
  강원도 영월군: 10건
  페이지 1: 26건  (누적 26/26)
  강원도 원주시: 26건
  페이지 1: 7건  (누적 7/7)
  강원도 인제군: 7건
  페이지 1: 10건  (누적 10/10)
  강원도 정선군: 10건
  페이지 1: 8건  (누적 8/8)
  강원도 철원군: 8건
  페이지 1: 26건  (누적 26/26)
  강원도 춘천시: 26건
  페이지 1: 9건  (누적 9/9)
  강원도 태백시: 9건
  페이지 1: 9건  (누적 9/9)
  강원도 평창군: 9건
  ⚠️  API 오류: 데이터 정보가 없습니다. 입력 파라미터값을 확인해주시기 바랍니다.
  페이지 1: 11건  (누적 11/11)
  강원도 홍천군: 11건
  페이지 1: 6건  (누적 6/6)
  강원도 화천군: 6건
  페이지 1: 10건  (누적 10/10)
  강원도 횡성군: 10건
  페이지 1: 16건  (누적 16/16)
  페이지 1: 15건  (누적 15/15)
  전라북도 고창군: 15건
  페이지 1: 28건  (누적 28/28)
  전라북도 군산시: 28건
  페이지 1: 20건  (누적 20/20)
  전라북도 김제시: 20건
  페이지 1: 24건  (누적 24/24)
  전라북도 남원시: 24건
  페이지 1: 7건  (누적 7/7)
  전라북도 무주군: 7건
  페이지 1: 14건  